In [1]:
import pandas as pd
import re
import numpy as np


input_file = "doaj_articles.csv"
output_file = "doaj_articles_fixed.csv"


def clean_text(text):
    """
    Funkcja czyszcząca tekst:
    1. Spłaszcza rozjechane wiersze (usuwa entery i wielokrotne spacje).
    2. Usuwa tagi HTML (<...>).
    3. Usuwa ewentualne ucięte tagi jats (bezpiecznie dla cudzysłowów).
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie (zamiana enterów i tabulatorów na spację)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie standardowych tagów (<i>, </i>, <p> itp.)
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Dodatkowe zabezpieczenie na "dziwne" tagi (jak w CORE)
    text = re.sub(r'</?jats:\w+>?', '', text)
    
    return text

print(f"Wczytywanie pliku {input_file}...")
try:
    df = pd.read_csv(input_file, on_bad_lines='skip')
    print(f"Wczytano {len(df):,} wierszy.")
except Exception as e:
    print(f"Błąd krytyczny przy wczytywaniu: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 1. Czyszczenie tekstów (Title, Abstract)
    print("Rozpoczynam czyszczenie tagów i spłaszczanie...")
    cols_to_clean = ['title', 'abstract', 'keywords', 'creator']
    
    for col in cols_to_clean:
        if col in df.columns:
            ## konwersja na string i czyszczenie
            df[col] = df[col].astype(str).apply(clean_text)
            ## zamiana pustych napisów i słowa 'nan' na prawdziwą pustkę (NaN)
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
            df[col] = df[col].replace('nan', np.nan)

    # 2. Uzupełnianie pustych keywords
    print("Uzupełnianie brakujących keywords wartościami z query...")
    if 'query' in df.columns and 'keywords' in df.columns:
        missing_count = df['keywords'].isna().sum()
        df['keywords'] = df['keywords'].fillna(df['query'])
        print(f"Uzupełniono {missing_count:,} pustych pól w 'keywords'.")

    # 3. Usuwanie śmieci (tylko gdy brak tytułu i abstraktu jednocześnie)
    print("Usuwanie pustych rekordów (bez tytułu i abstraktu)...")
    before_drop = len(df)
    ### how='all' oznacza: usuń wiersz tylko, jeśli wszystkie wskazane kolumny są puste (NaN)
    df = df.dropna(subset=['title', 'abstract'], how='all')
    after_drop = len(df)
    if before_drop > after_drop:
        print(f"Usunięto {before_drop - after_drop:,} kompletnie pustych wierszy.")

    # 4. Deduplikacja
    dedup_cols = ['title', 'abstract']
    print(f"Deduplikacja wg kolumn: {dedup_cols}...")
    
    initial_count = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    final_count = len(df)
    
    print(f"Usunięto duplikatów: {initial_count - final_count:,}")
    print(f"Liczba unikalnych rekordów: {final_count:,}")

    # 5. Zapis
    print(f"Zapisywanie do {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd wyniku
    print("\nPodgląd próbki danych")
    if 'year' in df.columns:
        print(df[['title', 'year']].head(3))
    else:
        print(df[['title']].head(3))
        
    if 'abstract' in df.columns and not df['abstract'].dropna().empty:
        print("\nPrzykładowy abstrakt (czysty):")
        print(df['abstract'].dropna().iloc[0][:200] + "...")


Wczytywanie pliku doaj_articles.csv...
Wczytano 5,771 wierszy.
Rozpoczynam czyszczenie tagów i spłaszczanie...
Uzupełnianie brakujących keywords wartościami z query...
Uzupełniono 1,109 pustych pól w 'keywords'.
Usuwanie pustych rekordów (bez tytułu i abstraktu)...
Deduplikacja wg kolumn: ['title', 'abstract']...
Usunięto duplikatów: 616
Liczba unikalnych rekordów: 5,155
Zapisywanie do doaj_articles_fixed.csv...

Podgląd próbki danych
                                               title    year
0  Baltic Sea Coastal Eutrophication in a Thousan...  2019.0
1  Early pastoral economies along the Ancient Sil...  2018.0
2  The figure of the impostor to the throne in Ru...  2017.0

Przykładowy abstrakt (czysty):
Sediment cores from three sites along the east-coast of Sweden, north-western Baltic Proper, have been studied with respect to lithologies, geochemistry, and diatom assemblages to trace and date early...


# update

In [2]:
import pandas as pd
import numpy as np
import re

input_file = "doaj_articles_fixed.csv"
output_file = input_file

def super_clean(text):
    """Jeszcze silniejsze spłaszczanie (usuwa wszelkie entery, tabulatory, nadmiarowe spacje)"""
    if not isinstance(text, str):
        return text
    ## zamiana wszystkiego co białe (enter, tab) na spację + usunięcie spacji z brzegów
    return re.sub(r'\s+', ' ', text).strip()

print(f"Wczytywanie {input_file}...")
try:
    df = pd.read_csv(input_file)
    print(f"Wczytano {len(df):,} wierszy.")
except Exception as e:
    print(f"Błąd: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 1. Naprawa daty (float -> Int64)
    if 'year' in df.columns:
        print("Naprawianie formatu roku...")
        ### 'coerce' zamieni błędy na NaN, Int64 obsłuży NaN bez kropki .0
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # 2. Spłaszczanie wszystkich kolumn tekstowych (żeby usunąć entery z linków itp.) ### wybieranie kolumn typu object (napisy)
    text_cols = df.select_dtypes(include=['object']).columns
    print(f"Doczyszczanie kolumn: {list(text_cols)}...")
    
    for col in text_cols:
        ### aplikowanie czyszczenia
        df[col] = df[col].apply(super_clean)
        
    # 3. Zapis finalny
    print(f"Zapisywanie do {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd
    print("\nPróbka po ostatecznym liftingu")
    print(df[['title', 'year']].head(3))
    
    # sprawdzenie czy linki są czyste (bez enterów)
    if 'link' in df.columns:
        print("\nPrzykładowy link:")
        print(df['link'].iloc[0])


Wczytywanie doaj_articles_fixed.csv...
Wczytano 5,155 wierszy.
Naprawianie formatu roku...
Doczyszczanie kolumn: ['title', 'abstract', 'keywords', 'journal_title', 'link', 'query']...
Zapisywanie do doaj_articles_fixed.csv...

Próbka po ostatecznym liftingu
                                               title  year
0  Baltic Sea Coastal Eutrophication in a Thousan...  2019
1  Early pastoral economies along the Ancient Sil...  2018
2  The figure of the impostor to the throne in Ru...  2017

Przykładowy link:
https://www.frontiersin.org/article/10.3389/fenvs.2019.00088/full
